In [1]:
from pathlib import Path
import pandas as pd
import sys

print("Arbeitsordner:", Path.cwd())
print("Python:", sys.executable)
print("Pandas-Version:", pd.__version__)

Arbeitsordner: /Users/saschagolparvari
Python: /Users/saschagolparvari/m5-umgebung/bin/python
Pandas-Version: 3.0.6


In [5]:
import subprocess
import sys
from pathlib import Path

projekt = Path.home() / "Desktop" / "Modul 5" / "Aufgaben" / "Woche_3"

ergebnis = subprocess.run(
    [sys.executable, "pipeline_skelett.py", "ingest"],
    cwd=projekt,
    capture_output=True,
    text=True
)

print("AUSGABE:")
print(ergebnis.stdout)

print("FEHLER:")
print(ergebnis.stderr)

print("Exit-Code:", ergebnis.returncode)

AUSGABE:
Quelle: bestellungen_april_roh.csv
Ziel: artefakte/01_ingest.parquet
Zeilenanzahl: 80

FEHLER:

Exit-Code: 0


In [6]:
original = pd.read_csv(projekt / "bestellungen_april_roh.csv")

gespeichert = pd.read_parquet(
    projekt / "artefakte" / "01_ingest.parquet"
)

pd.testing.assert_frame_equal(original, gespeichert)

print("Prüfung bestanden!")
print("Original:", len(original), "Zeilen")
print("Parquet:", len(gespeichert), "Zeilen")
print("Erkannte Spalten:", gespeichert.columns.tolist())

Prüfung bestanden!
Original: 80 Zeilen
Parquet: 80 Zeilen
Erkannte Spalten: ['bestell_id', 'datum', 'kunde_email', 'land', 'produkt', 'menge', 'preis_eur']


In [7]:
print("=== SPALTEN ===")
print(gespeichert.columns.tolist())

print("\n=== ERSTE 5 BESTELLUNGEN ===")
display(gespeichert.head())

print("\n=== FEHLENDE WERTE ===")
print(gespeichert.isna().sum())

print("\n=== EXAKTE DUPLIKATE ===")
print(gespeichert.duplicated().sum())

=== SPALTEN ===
['bestell_id', 'datum', 'kunde_email', 'land', 'produkt', 'menge', 'preis_eur']

=== ERSTE 5 BESTELLUNGEN ===


,bestell_id,datum,kunde_email,land,produkt,menge,preis_eur
0,3001,2026-04-11,NaN,DE,Dockingstation,2,"9,95"
1,3002,2026-04-02,umut.43@web.de,DE,Maus,2,"9,95"
2,3003,2026-04-02,olivia.31@gmx.at,DE,Tastatur,4,"19,90"
3,3004,11.04.2026,rosa.05@gmail.com,Österreich,Headset,3,"19,90"
4,3005,13.04.2026,mia.92@gmail.com,DE,Dockingstation,3,59



=== FEHLENDE WERTE ===
bestell_id     0
datum          0
kunde_email    4
land           0
produkt        0
menge          0
preis_eur      0
dtype: int64

=== EXAKTE DUPLIKATE ===
6


In [8]:
bereinigt = gespeichert.drop_duplicates().copy()

print("=== DUPLIKATBEREINIGUNG ===")
print("Vorher:", len(gespeichert), "Zeilen")
print("Nachher:", len(bereinigt), "Zeilen")
print("Entfernt:", len(gespeichert) - len(bereinigt))
print("Verbleibende Duplikate:", bereinigt.duplicated().sum())

=== DUPLIKATBEREINIGUNG ===
Vorher: 80 Zeilen
Nachher: 74 Zeilen
Entfernt: 6
Verbleibende Duplikate: 0


In [9]:
print("=== DATUMSFORMATE ===")

print("ISO-Format (JJJJ-MM-TT):")
print(bereinigt["datum"].str.fullmatch(r"\d{4}-\d{2}-\d{2}").sum())

print("\nDeutsches Format (TT.MM.JJJJ):")
print(bereinigt["datum"].str.fullmatch(r"\d{2}\.\d{2}\.\d{4}").sum())

print("\nBeispielwerte:")
print(bereinigt["datum"].unique()[:10])

=== DATUMSFORMATE ===
ISO-Format (JJJJ-MM-TT):
55

Deutsches Format (TT.MM.JJJJ):
19

Beispielwerte:
<ArrowStringArray>
['2026-04-11', '2026-04-02', '11.04.2026', '13.04.2026', '16.04.2026',
 '2026-04-06', '2026-04-22', '2026-04-01', '19.04.2026', '2026-04-16']
Length: 10, dtype: str


In [10]:
# Die beiden Datumsformate erkennen
iso = bereinigt["datum"].str.fullmatch(r"\d{4}-\d{2}-\d{2}")
de = bereinigt["datum"].str.fullmatch(r"\d{2}\.\d{2}\.\d{4}")

# Leere Datumsspalte vorbereiten
datum_neu = pd.Series(pd.NaT, index=bereinigt.index, dtype="datetime64[ns]")

# ISO-Daten umwandeln
datum_neu.loc[iso] = pd.to_datetime(
    bereinigt.loc[iso, "datum"],
    format="%Y-%m-%d"
)

# Deutsche Daten umwandeln
datum_neu.loc[de] = pd.to_datetime(
    bereinigt.loc[de, "datum"],
    format="%d.%m.%Y"
)

# Bereinigte Datumswerte übernehmen
bereinigt["datum"] = datum_neu

print("Fehlerhafte Datumswerte:", bereinigt["datum"].isna().sum())
print("Frühestes Datum:", bereinigt["datum"].min())
print("Spätestes Datum:", bereinigt["datum"].max())

Fehlerhafte Datumswerte: 0
Frühestes Datum: 2026-04-01 00:00:00
Spätestes Datum: 2026-04-30 00:00:00


In [11]:
print("=== PREISANALYSE ===")

print("Aktueller Datentyp:", bereinigt["preis_eur"].dtype)

print("\nBeispielpreise:")
print(bereinigt["preis_eur"].unique()[:15])

print("\nPreise mit Komma:")
print(bereinigt["preis_eur"].str.contains(",", regex=False).sum())

=== PREISANALYSE ===
Aktueller Datentyp: str

Beispielpreise:
<ArrowStringArray>
['9,95', '19,90', '59', '149', '39,90', '129']
Length: 6, dtype: str

Preise mit Komma:
42


In [12]:
# Dezimalkomma durch Dezimalpunkt ersetzen
bereinigt["preis_eur"] = (
    bereinigt["preis_eur"]
    .str.replace(",", ".", regex=False)
    .astype(float)
)

print("=== PREISBEREINIGUNG ===")
print("Neuer Datentyp:", bereinigt["preis_eur"].dtype)
print("Fehlende Preise:", bereinigt["preis_eur"].isna().sum())
print("Beispielpreise:")
print(bereinigt["preis_eur"].unique())

=== PREISBEREINIGUNG ===
Neuer Datentyp: float64
Fehlende Preise: 0
Beispielpreise:
[  9.95  19.9   59.   149.    39.9  129.  ]


In [13]:
print("=== LÄNDERANALYSE ===")

print(bereinigt["land"].value_counts(dropna=False))

print("\nUnterschiedliche Schreibweisen:")
print(bereinigt["land"].unique())

=== LÄNDERANALYSE ===
land
DE             24
AT             17
de             12
at             11
Österreich      6
Deutschland     4
Name: count, dtype: int64

Unterschiedliche Schreibweisen:
<ArrowStringArray>
['DE', 'Österreich', 'at', 'de', 'AT', 'Deutschland']
Length: 6, dtype: str


In [14]:
# Einheitliche Ländercodes festlegen
land_mapping = {
    "DE": "DE",
    "de": "DE",
    "Deutschland": "DE",
    "AT": "AT",
    "at": "AT",
    "Österreich": "AT"
}

# Länderbezeichnungen vereinheitlichen
bereinigt["land"] = bereinigt["land"].map(land_mapping)

print("=== LÄNDERBEREINIGUNG ===")
print(bereinigt["land"].value_counts())

print("\nUnbekannte Länder:", bereinigt["land"].isna().sum())
print("Gesamtzahl:", len(bereinigt))

=== LÄNDERBEREINIGUNG ===
land
DE    40
AT    34
Name: count, dtype: int64

Unbekannte Länder: 0
Gesamtzahl: 74


In [15]:
print("=== BESTELLUNGEN OHNE E-MAIL ===")

ohne_email = bereinigt[
    bereinigt["kunde_email"].isna()
]

display(ohne_email)

print("Betroffene Bestellungen:", len(ohne_email))

=== BESTELLUNGEN OHNE E-MAIL ===


,bestell_id,datum,kunde_email,land,produkt,menge,preis_eur
0,3001,2026-04-11,NaN,DE,Dockingstation,2,9.95
20,3020,2026-04-28,NaN,DE,USB-C-Hub,3,9.95
22,3022,2026-04-23,NaN,AT,Maus,2,59.00
53,3049,2026-04-26,NaN,AT,Headset,1,39.90


Betroffene Bestellungen: 4


In [16]:
ergebnis = subprocess.run(
    [sys.executable, "pipeline_skelett.py", "clean"],
    cwd=projekt,
    capture_output=True,
    text=True
)

print("=== AUSGABE ===")
print(ergebnis.stdout)

print("=== FEHLER ===")
print(ergebnis.stderr)

print("Exit-Code:", ergebnis.returncode)

=== AUSGABE ===
=== DUPLIKATBEREINIGUNG ===
Vorher: 80
Nachher: 74
Entfernt: 6
Fehlerhafte Datumswerte: 0
Frühestes Datum: 2026-04-01 00:00:00
Spätestes Datum: 2026-04-30 00:00:00
=== PREISBEREINIGUNG ===
Datentyp: float64
Fehlende Preise: 0
=== LÄNDERBEREINIGUNG ===
land
DE    40
AT    34
Name: count, dtype: int64
Unbekannte Länder: 0
=== E-MAIL-PRÜFUNG ===
Fehlende E-Mail-Adressen: 4
Entscheidung: Bestellungen bleiben erhalten.
=== QUALITÄTSPRÜFUNG ===
Alle drei Prüfungen bestanden!
=== CLEAN ABGESCHLOSSEN ===
Gespeicherte Bestellungen: 74
Zieldatei: artefakte/02_clean.parquet

=== FEHLER ===

Exit-Code: 0


In [17]:
clean_datei = projekt / "artefakte" / "02_clean.parquet"

clean_df = pd.read_parquet(clean_datei)

print("=== GESPEICHERTE DATEN ===")
print("Zeilen:", len(clean_df))
print("Spalten:", len(clean_df.columns))
print("Duplikate:", clean_df.duplicated().sum())

print("\nDatentypen:")
print(clean_df.dtypes)

display(clean_df.head()) 

=== GESPEICHERTE DATEN ===
Zeilen: 74
Spalten: 7
Duplikate: 0

Datentypen:
bestell_id              int64
datum          datetime64[ns]
kunde_email               str
land                      str
produkt                   str
menge                   int64
preis_eur             float64
dtype: object


,bestell_id,datum,kunde_email,land,produkt,menge,preis_eur
0,3001,2026-04-11,NaN,DE,Dockingstation,2,9.95
1,3002,2026-04-02,umut.43@web.de,DE,Maus,2,9.95
2,3003,2026-04-02,olivia.31@gmx.at,DE,Tastatur,4,19.90
3,3004,2026-04-11,rosa.05@gmail.com,AT,Headset,3,19.90
4,3005,2026-04-13,mia.92@gmail.com,DE,Dockingstation,3,59.00


In [18]:
ergebnis = subprocess.run(
    [sys.executable, "pipeline_skelett.py", "store"],
    cwd=projekt,
    capture_output=True,
    text=True
)

print("=== AUSGABE ===")
print(ergebnis.stdout)

print("=== FEHLER ===")
print(ergebnis.stderr)

print("Exit-Code:", ergebnis.returncode)

=== AUSGABE ===
=== STORE ABGESCHLOSSEN ===
Quelle: artefakte/02_clean.parquet
Datenbank: artefakte/shop.db
Gespeicherte Bestellungen: 74

=== FEHLER ===

Exit-Code: 0


In [19]:
ergebnis = subprocess.run(
    [sys.executable, "pipeline_skelett.py", "all"],
    cwd=projekt,
    capture_output=True,
    text=True
)

print("=== GESAMTE PIPELINE ===")
print(ergebnis.stdout)

print("=== FEHLER ===")
print(ergebnis.stderr)

print("Exit-Code:", ergebnis.returncode)

=== GESAMTE PIPELINE ===
Quelle: bestellungen_april_roh.csv
Ziel: artefakte/01_ingest.parquet
Zeilenanzahl: 80
=== DUPLIKATBEREINIGUNG ===
Vorher: 80
Nachher: 74
Entfernt: 6
Fehlerhafte Datumswerte: 0
Frühestes Datum: 2026-04-01 00:00:00
Spätestes Datum: 2026-04-30 00:00:00
=== PREISBEREINIGUNG ===
Datentyp: float64
Fehlende Preise: 0
=== LÄNDERBEREINIGUNG ===
land
DE    40
AT    34
Name: count, dtype: int64
Unbekannte Länder: 0
=== E-MAIL-PRÜFUNG ===
Fehlende E-Mail-Adressen: 4
Entscheidung: Bestellungen bleiben erhalten.
=== QUALITÄTSPRÜFUNG ===
Alle drei Prüfungen bestanden!
=== CLEAN ABGESCHLOSSEN ===
Gespeicherte Bestellungen: 74
Zieldatei: artefakte/02_clean.parquet
=== STORE ABGESCHLOSSEN ===
Quelle: artefakte/02_clean.parquet
Datenbank: artefakte/shop.db
Gespeicherte Bestellungen: 74
[lauf] ok - alle Artefakte in artefakte/

=== FEHLER ===

Exit-Code: 0


 ## Woche 3 – Tag 1: Pipeline-Verträge

### Ablauf der ETL-Pipeline

| Schritt | Eingabe | Artefakt | Nachweis |
|---|---|---|---|
| ingest() | bestellungen_april_roh.csv | artefakte/01_ingest.parquet | 80 Zeilen, 7 Spalten, Originalvergleich bestanden |
| clean() | artefakte/01_ingest.parquet | artefakte/02_clean.parquet | 74 Zeilen, 6 Duplikate entfernt, Qualitätsprüfungen bestanden |
| store() | artefakte/02_clean.parquet | artefakte/shop.db | 74 Bestellungen in SQLite gespeichert |

### Ergebnis des Gesamtlaufs

Die Pipeline wurde mit `python pipeline_skelett.py all`
vollständig ausgeführt.

- Rohdaten: 80 Bestellungen
- Entfernte Duplikate: 6
- Bereinigte Bestellungen: 74
- Fehlende E-Mail-Adressen: 4 (bewusst beibehalten)
- Ungültige Datumswerte: 0
- Unbekannte Länder: 0
- SQLite-Bestellungen: 74
- Exit-Code: 0

Alle drei Pipeline-Schritte wurden erfolgreich ausgeführt.

### KI-Reflexion

Für diese Aufgabe habe ich ChatGPT als Lern- und Programmierunterstützung verwendet; GitHub Copilot kam nicht zum Einsatz. ChatGPT half mir dabei, die drei Pipeline-Funktionen schrittweise zu implementieren und die einzelnen Bereinigungsregeln zu verstehen. Beim Testen fiel auf, dass die CSV-Datei zunächst mit dem falschen Trennzeichen eingelesen wurde; durch die eigenständige Kontrolle der Spalten und Daten habe ich diesen Fehler erkannt und anschließend mit Unterstützung von ChatGPT korrigiert.